<a href="https://colab.research.google.com/github/suryanshshah2006/Sleep-stage-classification-using-EEG-data-/blob/main/Sleep_Stage_classification_using_EEG_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install mne pyedflib openneuro-py

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!wget -q -O participants.tsv \
https://github.com/OpenNeuroDatasets/ds007181/raw/main/participants.tsv

In [ ]:
!rm -rf /content/ds007181_meta
!git clone --depth 1 --filter=blob:none --sparse https://github.com/OpenNeuroDatasets/ds007181.git /content/ds007181_meta

In [ ]:
%cd /content/ds007181_meta

!git sparse-checkout init --no-cone

!git sparse-checkout set \
  participants.tsv \
  "sub-*/eeg/*_channels.tsv" \
  "sub-*/eeg/*_events.tsv"

!git checkout

In [ ]:
!pip install mne matplotlib pandas

from google.colab import drive
import os
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
import pandas as pd
import os
import glob

chunks_folder = '/content/drive/MyDrive/EEG_Full_58_Subjects'
meta_base_path = '/content/ds007181_meta'

label_mapping = {
    'W': 0, 'Sleep stage W': 0,
    'N1': 1, 'Sleep stage N1': 1,
    'N2': 2, 'Sleep stage N2': 2,
    'N3': 3, 'Sleep stage N3': 3,
    'R': 4, 'REM': 4, 'Sleep stage R': 4
}

all_labels = []

processed_files = glob.glob(os.path.join(chunks_folder, '*_seg_0.npy'))
subjects = [os.path.basename(f).split('_')[0] for f in processed_files]
print(f"Generating labels for {len(subjects)} subjects...")

for sub in subjects:
    tsv_path = os.path.join(meta_base_path, sub, 'eeg', f'{sub}_task-sleep_acq-PSG_events.tsv')

    if not os.path.exists(tsv_path):
        tsv_path = os.path.join(meta_base_path, f'{sub}_task-sleep_acq-PSG_events.tsv')
        if not os.path.exists(tsv_path):
            continue

    events_df = pd.read_csv(tsv_path, sep='\t')

    if 'stage' in events_df.columns: label_col = 'stage'
    elif 'value' in events_df.columns: label_col = 'value'
    elif 'trial_type' in events_df.columns: label_col = 'trial_type'
    else: continue

    epoch_duration = 30

    for index, row in events_df.iterrows():
        onset = row['onset']
        duration = row['duration']
        label = row[label_col]

        start_chunk_idx = int(onset // epoch_duration)
        num_chunks = int(duration // epoch_duration)

        for i in range(num_chunks):
            chunk_idx = start_chunk_idx + i
            filename = f"{sub}_seg_{chunk_idx}.npy"
            filepath = os.path.join(chunks_folder, filename)

            all_labels.append({
                'filename': filename,
                'label': label,
                'filepath': filepath,
                'subject': sub
            })

master_df = pd.DataFrame(all_labels)
master_df['label_code'] = master_df['label'].map(label_mapping)
master_df = master_df.dropna(subset=['label_code'])

print(f"Total labeled samples: {len(master_df)}")
master_df.to_csv('/content/drive/MyDrive/EEG_Full_58_Subjects/master_labels_full.csv', index=False)

In [ ]:
import pandas as pd
import os
import glob

PROCESSED_PATH = '/content/drive/MyDrive/EEG_Full_58_Subjects'
META_PATH = '/content/ds007181_meta'
OUTPUT_CSV = os.path.join(PROCESSED_PATH, 'master_labels_clean.csv')

label_mapping = {
    'W': 0, 'Sleep stage W': 0,
    'N1': 1, 'Sleep stage N1': 1,
    'N2': 2, 'Sleep stage N2': 2,
    'N3': 3, 'Sleep stage N3': 3,
    'R': 4, 'REM': 4, 'Sleep stage R': 4
}

print("1. Scanning for processed files on Drive...")
processed_files = glob.glob(os.path.join(PROCESSED_PATH, '*_seg_0.npy'))
subjects = sorted(list(set([os.path.basename(f).split('_')[0] for f in processed_files])))
print(f"✅ Found {len(subjects)} subjects with data.")

all_labels = []

print("2. Matching files to labels (This takes ~60 seconds)...")
for sub in subjects:
    tsv_path = os.path.join(META_PATH, sub, 'eeg', f'{sub}_task-sleep_acq-PSG_events.tsv')
    if not os.path.exists(tsv_path):
        tsv_path = os.path.join(META_PATH, f'{sub}_task-sleep_acq-PSG_events.tsv')
        if not os.path.exists(tsv_path): continue

    try:
        events_df = pd.read_csv(tsv_path, sep='\t')
    except: continue

    col = next((c for c in ['stage', 'value', 'trial_type'] if c in events_df.columns), None)
    if not col: continue

    for _, row in events_df.iterrows():
        onset = row['onset']
        duration = row['duration']
        label = row[col]

        start_chunk = int(onset // 30)
        num_chunks = int(duration // 30)

        for i in range(num_chunks):
            fname = f"{sub}_seg_{start_chunk + i}.npy"
            fpath = os.path.join(PROCESSED_PATH, fname)

            all_labels.append({
                'filename': fname,
                'label': label,
                'filepath': fpath,
                'subject': sub
            })

master_df = pd.DataFrame(all_labels)
master_df['label_code'] = master_df['label'].map(label_mapping)
master_df = master_df.dropna(subset=['label_code'])

print(f"3. Verifying file existence for {len(master_df)} samples...")
master_df['exists'] = master_df['filepath'].apply(os.path.exists)
clean_df = master_df[master_df['exists'] == True].copy()

clean_df.to_csv(OUTPUT_CSV, index=False)
print(f"\n🎉 SUCCESS! Saved clean list to: {OUTPUT_CSV}")
print(f"Total valid samples: {len(clean_df)}")

In [ ]:
import os
import shutil
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from tqdm.notebook import tqdm

SOURCE_DIR = '/content/drive/MyDrive/EEG_Full_58_Subjects'
DEST_DIR = '/content/local_eeg_data'
DOWNSAMPLE = 10

TARGET_SUBS = [f'sub-{i:02d}' for i in range(1, 60)]

print("🧹 Wiping old local data to make room...")
if os.path.exists(DEST_DIR):
    shutil.rmtree(DEST_DIR)
os.makedirs(DEST_DIR, exist_ok=True)

print("🔍 Scanning Drive for ALL subjects...")
all_files = os.listdir(SOURCE_DIR)

files_to_copy = [f for f in all_files if any(f.startswith(sub) for sub in TARGET_SUBS)]
print(f"✅ Found {len(files_to_copy)} total files. (Expected: ~47,000)")

def smart_copy(filename):
    try:
        src = os.path.join(SOURCE_DIR, filename)
        data = np.load(src)
        data = data[:, ::DOWNSAMPLE].astype(np.float16)
        dst = os.path.join(DEST_DIR, filename)
        np.save(dst, data)
    except Exception:
        pass

print(f"🚀 Copying & Compressing {len(files_to_copy)} files... (Go grab a coffee, this takes ~45 mins)")
with ThreadPoolExecutor(max_workers=8) as executor:
    list(tqdm(executor.map(smart_copy, files_to_copy), total=len(files_to_copy)))

print("\n🎉 MASSIVE Transfer Complete! You now have the full dataset locally.")

In [ ]:
import pandas as pd
import numpy as np
import os
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

print("1. Syncing Master List with Local Files...")
LOCAL_DIR = '/content/local_eeg_data'
CSV_PATH = '/content/drive/MyDrive/EEG_Full_58_Subjects/master_labels_clean.csv'

master_df = pd.read_csv(CSV_PATH)

local_files = set(os.listdir(LOCAL_DIR))
print(f"Found {len(local_files)} files in local directory.")

local_df = master_df[master_df['filename'].isin(local_files)].copy()

local_df['filepath'] = local_df['filename'].apply(lambda x: os.path.join(LOCAL_DIR, x))

print(f"✅ Filtered down to {len(local_df)} matched samples ready for massive training.")

train_df, val_df = train_test_split(
    local_df, test_size=0.2, random_state=42, stratify=local_df['label_code']
)
print(f"✅ Split complete: {len(train_df)} Train | {len(val_df)} Val")

class EEGDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, df, batch_size=64, n_classes=5):
        self.df = df
        self.batch_size = batch_size
        self.n_classes = n_classes
        self.indices = np.arange(len(self.df))

    def __len__(self):
        return int(np.floor(len(self.df) / self.batch_size))

    def __getitem__(self, index):
        batch_indices = self.indices[index*self.batch_size : (index+1)*self.batch_size]
        batch_df = self.df.iloc[batch_indices]
        X, y = [], []
        for _, row in batch_df.iterrows():
            data = np.load(row['filepath']).astype(np.float32)

            mean, std = np.mean(data), np.std(data)
            data = (data - mean) / std if std > 0 else data - mean

            X.append(data)
            y.append(row['label_code'])
        return np.array(X), to_categorical(np.array(y), num_classes=self.n_classes)

    def on_epoch_end(self):
        np.random.shuffle(self.indices)

train_gen = EEGDataGenerator(train_df, batch_size=64)
val_gen = EEGDataGenerator(val_df, batch_size=64)

print("\n2. Configuring Model Shape...")
sample_shape = np.load(train_df.iloc[0]['filepath']).shape
n_channels = sample_shape[0]
n_points = sample_shape[1]
print(f"✅ Input Shape: Channels={n_channels}, Timepoints={n_points}")
print("\n3. Building Model...")
model = tf.keras.Sequential([
    tf.keras.layers.InputLayer(shape=(n_channels, n_points)),
    tf.keras.layers.Permute((2, 1)),

    tf.keras.layers.Conv1D(32, 64, padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling1D(4),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Conv1D(64, 32, padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling1D(4),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=False)),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(5, activation='softmax')
])

optimizer = tf.keras.optimizers.Adam(learning_rate=0.0005)
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

y_train = train_df['label_code'].values
weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)

print("\n🚀 Starting Fast Training on Full Dataset for 50 Epochs...")
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=50,
    class_weight=dict(enumerate(weights)),
    callbacks=[
        ModelCheckpoint('/content/drive/MyDrive/sleep_model_bilstm_FULL.keras', save_best_only=True, monitor='val_accuracy'),
        EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    ]
)

In [ ]:
import pandas as pd
import numpy as np
import os
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

print("1. Syncing Master List with Local Files...")
LOCAL_DIR = '/content/local_eeg_data'
CSV_PATH = '/content/drive/MyDrive/EEG_Full_58_Subjects/master_labels_clean.csv'

master_df = pd.read_csv(CSV_PATH)

local_files = set(os.listdir(LOCAL_DIR))
print(f"Found {len(local_files)} files in local directory.")

local_df = master_df[master_df['filename'].isin(local_files)].copy()

local_df['filepath'] = local_df['filename'].apply(lambda x: os.path.join(LOCAL_DIR, x))

print(f"✅ Filtered down to {len(local_df)} matched samples ready for massive training.")

train_df, val_df = train_test_split(
    local_df, test_size=0.2, random_state=42, stratify=local_df['label_code']
)
print(f"✅ Split complete: {len(train_df)} Train | {len(val_df)} Val")

class EEGDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, df, batch_size=64, n_classes=5):
        self.df = df
        self.batch_size = batch_size
        self.n_classes = n_classes
        self.indices = np.arange(len(self.df))

    def __len__(self):
        return int(np.floor(len(self.df) / self.batch_size))

    def __getitem__(self, index):
        batch_indices = self.indices[index*self.batch_size : (index+1)*self.batch_size]
        batch_df = self.df.iloc[batch_indices]
        X, y = [], []
        for _, row in batch_df.iterrows():
            data = np.load(row['filepath']).astype(np.float32)
            mean, std = np.mean(data), np.std(data)
            data = (data - mean) / std if std > 0 else data - mean
            X.append(data)
            y.append(row['label_code'])
        return np.array(X), to_categorical(np.array(y), num_classes=self.n_classes)

    def on_epoch_end(self):
        np.random.shuffle(self.indices)

train_gen = EEGDataGenerator(train_df, batch_size=64)
val_gen = EEGDataGenerator(val_df, batch_size=64)

print("\n2. Configuring Model Shape...")
sample_shape = np.load(train_df.iloc[0]['filepath']).shape
n_channels = sample_shape[0]
n_points = sample_shape[1]
print(f"✅ Input Shape: Channels={n_channels}, Timepoints={n_points}")
-
print("\n3. Building SOTA Multi-Scale Model...")
inputs = tf.keras.layers.Input(shape=(n_channels, n_points))
x = tf.keras.layers.Permute((2, 1))(inputs) # (Time, Channels)

conv_fine = tf.keras.layers.Conv1D(64, kernel_size=32, strides=4, padding='same', activation='relu')(x)
conv_fine = tf.keras.layers.BatchNormalization()(conv_fine)
conv_fine = tf.keras.layers.MaxPooling1D(4)(conv_fine)

conv_coarse = tf.keras.layers.Conv1D(64, kernel_size=200, strides=4, padding='same', activation='relu')(x)
conv_coarse = tf.keras.layers.BatchNormalization()(conv_coarse)
conv_coarse = tf.keras.layers.MaxPooling1D(4)(conv_coarse)

merged_cnn = tf.keras.layers.Concatenate(axis=-1)([conv_fine, conv_coarse])
merged_cnn = tf.keras.layers.Dropout(0.4)(merged_cnn)

lstm_out = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=True))(merged_cnn)
lstm_out = tf.keras.layers.Dropout(0.4)(lstm_out)

attention_out = tf.keras.layers.Attention()([lstm_out, lstm_out])
pooled = tf.keras.layers.GlobalAveragePooling1D()(attention_out)

dense = tf.keras.layers.Dense(64, activation='relu')(pooled)
dense = tf.keras.layers.Dropout(0.5)(dense)
outputs = tf.keras.layers.Dense(5, activation='softmax')(dense)

model = tf.keras.Model(inputs=inputs, outputs=outputs)

print("\n🚀 Starting SOTA Training for 50 Epochs...")

focal_loss = tf.keras.losses.CategoricalFocalCrossentropy(alpha=0.25, gamma=2.0)
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0005)

model.compile(optimizer=optimizer, loss=focal_loss, metrics=['accuracy'])

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=50,
    callbacks=[
        ModelCheckpoint('/content/drive/MyDrive/sleep_model_SOTA.keras', save_best_only=True, monitor='val_accuracy'),
        EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    ]
)

In [ ]:
print("\n2. Configuring Model Shape...")
sample_data = np.load(train_df.iloc[0]['filepath'])
n_channels = sample_data.shape[0]
n_points = sample_data.shape[1]
print(f"✅ Input Shape Detected: Channels={n_channels}, Timepoints={n_points}")

print("\n3. Building Memory-Safe Tuned Model...")
inputs = tf.keras.layers.Input(shape=(n_channels, n_points))

In [ ]:
import pandas as pd
import numpy as np
import os
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

print("1. Syncing Master List with Local Files...")
LOCAL_DIR = '/content/local_eeg_data'
CSV_PATH = '/content/drive/MyDrive/EEG_Full_58_Subjects/master_labels_clean.csv'

master_df = pd.read_csv(CSV_PATH)

local_files = set(os.listdir(LOCAL_DIR))
print(f"Found {len(local_files)} files in local directory.")

local_df = master_df[master_df['filename'].isin(local_files)].copy()

local_df['filepath'] = local_df['filename'].apply(lambda x: os.path.join(LOCAL_DIR, x))

print(f"✅ Filtered down to {len(local_df)} matched samples ready for massive training.")

train_df, val_df = train_test_split(
    local_df, test_size=0.2, random_state=42, stratify=local_df['label_code']
)
print(f"✅ Split complete: {len(train_df)} Train | {len(val_df)} Val")

class EEGDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, df, batch_size=64, n_classes=5):
        self.df = df
        self.batch_size = batch_size
        self.n_classes = n_classes
        self.indices = np.arange(len(self.df))

    def __len__(self):
        return int(np.floor(len(self.df) / self.batch_size))

    def __getitem__(self, index):
        batch_indices = self.indices[index*self.batch_size : (index+1)*self.batch_size]
        batch_df = self.df.iloc[batch_indices]
        X, y = [], []
        for _, row in batch_df.iterrows():
            data = np.load(row['filepath']).astype(np.float32)
            mean, std = np.mean(data), np.std(data)
            data = (data - mean) / std if std > 0 else data - mean
            X.append(data)
            y.append(row['label_code'])
        return np.array(X), to_categorical(np.array(y), num_classes=self.n_classes)

    def on_epoch_end(self):
        np.random.shuffle(self.indices)

train_gen = EEGDataGenerator(train_df, batch_size=32)
val_gen = EEGDataGenerator(val_df, batch_size=32)

print("\n3. Building Memory-Safe Tuned Model...")
inputs = tf.keras.layers.Input(shape=(n_channels, n_points))
x = tf.keras.layers.Permute((2, 1))(inputs)

s1 = tf.keras.layers.Conv1D(32, 32, strides=4, padding='same', activation='relu')(x)
s2 = tf.keras.layers.Conv1D(32, 128, strides=4, padding='same', activation='relu')(x)
merged = tf.keras.layers.Concatenate(axis=-1)([s1, s2])
merged = tf.keras.layers.BatchNormalization()(merged)
merged = tf.keras.layers.MaxPooling1D(4)(merged)
merged = tf.keras.layers.Dropout(0.3)(merged)

lstm_out = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=True))(merged)
attention_out = tf.keras.layers.Attention()([lstm_out, lstm_out])
pooled = tf.keras.layers.GlobalAveragePooling1D()(attention_out)

dense = tf.keras.layers.Dense(64, activation='relu')(pooled)
outputs = tf.keras.layers.Dense(5, activation='softmax')(dense)

model = tf.keras.Model(inputs=inputs, outputs=outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

lr_reducer = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)

print("\n🚀 Starting Safe Training...")
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=50,
    callbacks=[
        tf.keras.callbacks.ModelCheckpoint('/content/drive/MyDrive/sleep_model_SAFE_TUNED.keras',
                                           save_best_only=True, monitor='val_accuracy'),
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
        lr_reducer
    ]
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

print("\n📊 Evaluating Best Model (Restored Weights)...")
val_loss, val_acc = model.evaluate(val_gen)

train_loss, train_acc = model.evaluate(train_gen)

print(f"\n✅ Final Train Accuracy: {train_acc*100:.2f}%")
print(f"✅ Final Validation Accuracy: {val_acc*100:.2f}%")

In [ ]:
print("\n🔍 Running Locked Evaluation (No Shuffle)...")

class EvalGenerator(EEGDataGenerator):
    def __init__(self, df, batch_size=32, n_classes=5):
        super().__init__(df, batch_size, n_classes)
        self.indices = np.arange(len(self.df))
    def on_epoch_end(self):
        pass

eval_gen = EvalGenerator(val_df, batch_size=32)

y_true = []
y_pred_probs = []

for i in range(len(eval_gen)):
    X_batch, y_batch = eval_gen[i]
    y_true.extend(np.argmax(y_batch, axis=1))

    batch_preds = model.predict_on_batch(X_batch)
    y_pred_probs.extend(batch_preds)

y_true = np.array(y_true)
y_pred = np.argmax(np.array(y_pred_probs), axis=1)

class_labels = ['W', 'N1', 'N2', 'N3', 'REM']
print("\n✅ REAL Final Validation Accuracy:", np.mean(y_true == y_pred) * 100, "%")

print("\n📋 REAL Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_labels))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=class_labels, yticklabels=class_labels)
plt.title('REAL Confusion Matrix (Synced)')
plt.show()